# Power analysis (Track 4, step 1)

**Question:** how many matches are needed to detect a factor with rank-IC of 0.02 at a
Newey-West / match-clustered t-statistic of at least 2, as the draft success criteria
require? If that number is far above the matches any video source can supply, the
criteria must change *before* Stages 2 to 4 are built.

**Method: simulation.** Real data can't answer this (and must not be looked at before
pre-registration). Instead, fake matches are generated where the truth is known:
1. A slow-moving **factor** per match (think "sustained pressure by team A"),
   standardised, drifting minute by minute as an AR(1) process with persistence `phi`.
2. **Goals** for each team arrive at about 1.4 per team per match. The factor tilts
   them: team A's goal rate is multiplied by `exp(beta * factor)`, team B's by
   `exp(-beta * factor)`. So `beta = 0.1` means one standard deviation of the factor
   raises A's goal rate by about 10% and lowers B's by about 10%.
3. A **target**: goal difference (A minus B) over the next 5 minutes, next 15
   minutes, or the rest of the match, measured after the factor's timestamp (point in
   time correct).
4. The planned **test**: pooled rank-IC between factor and target, t-statistic with
   standard errors clustered by match (equivalent in spirit to Newey-West within a
   match: it allows any correlation between minutes of the same match).

Everything is simulated, nothing touches real data, so this is safe to run before
pre-registration. Assumptions are listed at the end.

In [1]:
import numpy as np
from scipy.stats import rankdata

MINUTES = 90
LAM0 = 1.4 / MINUTES          # goals per team per minute (about 2.8 per match in total)
T_CRIT = 2.0                  # success criterion: t >= 2
T_80 = T_CRIT + 0.8416        # mean t needed for 80% power (t is roughly normal, sd 1)
SEED = 20261005
rng = np.random.default_rng(SEED)


def simulate(n_matches, beta, phi, rng):
    # Factor: stationary AR(1) with unit variance, one value per minute.
    f = np.empty((n_matches, MINUTES))
    f[:, 0] = rng.standard_normal(n_matches)
    shocks = rng.standard_normal((n_matches, MINUTES)) * np.sqrt(1 - phi ** 2)
    for t in range(1, MINUTES):
        f[:, t] = phi * f[:, t - 1] + shocks[:, t]
    goals_a = rng.poisson(LAM0 * np.exp(beta * f))
    goals_b = rng.poisson(LAM0 * np.exp(-beta * f))
    return f, goals_a - goals_b, goals_a + goals_b


def target(diff, horizon):
    # Goal difference in minutes t+1 .. t+horizon (or to full time), for each minute t.
    c = np.concatenate([np.zeros((diff.shape[0], 1)), np.cumsum(diff, axis=1)], axis=1)
    if horizon == "rest":
        return c[:, [MINUTES]] - c[:, 1:MINUTES]
    return c[:, horizon + 1:] - c[:, 1:MINUTES - horizon + 1]


def rank_ic_test(f, y):
    # Pooled Spearman rank-IC, with match-clustered and naive t-statistics.
    n, T = y.shape
    rf = rankdata(f[:, :T].ravel())          # average ranks: goal targets have many ties
    ry = rankdata(y.ravel())
    rf = (rf - rf.mean()) / rf.std()
    ry = (ry - ry.mean()) / ry.std()
    ic = np.mean(rf * ry)
    resid = ry - ic * rf
    per_match = (rf * resid).reshape(n, T).sum(axis=1)
    var = np.sum(per_match ** 2) / np.sum(rf ** 2) ** 2 * n / (n - 1)
    t_cluster = ic / np.sqrt(var)
    t_naive = ic * np.sqrt(rf.size - 2) / np.sqrt(1 - ic ** 2)   # pretends minutes are independent
    return ic, t_cluster, t_naive


def hazard_z(f, diff, total):
    # Score test of beta = 0 in the goal-rate model itself, using the factor at minute
    # t-1 to explain goals in minute t. Disjoint minutes are independent given the
    # factor, so no autocorrelation correction is needed.
    u = np.sum(f[:, :-1] * diff[:, 1:])
    v = np.sum(f[:, :-1] ** 2 * total[:, 1:])
    return u / np.sqrt(v)


HORIZONS = (5, 15, "rest")
PHIS = (0.8, 0.95, 0.99)
print("ready")

ready


## Step 1: how much independent information does one match hold?

With no true effect (`beta = 0`), the spread of the IC across simulated datasets
shows how noisy the IC is. If minutes were independent, its standard deviation would
be `1 / sqrt(number of minutes)`. They aren't: the factor drifts slowly and the target
windows overlap, so neighbouring minutes repeat the same information.

`k` = **effective independent observations per match** = `1 / (n_matches * Var(IC))`.
A match has about 85 minute-rows but only `k` of them are effectively independent.

The same simulations check the test itself: with no effect, `t >= 2` should happen
about 2.3% of the time. The naive t (treating minutes as independent) shows why the
success criteria insist on Newey-West / clustered errors.

In [2]:
N0, SIMS = 200, 1000
k_table = {}
print(f"{'phi':>5} {'horizon':>8} {'sd(IC)':>8} {'k':>6}   false positives: clustered  naive")
for phi in PHIS:
    for h in HORIZONS:
        res = np.array([rank_ic_test(f, target(d, h))
                        for f, d, _ in (simulate(N0, 0.0, phi, rng) for _ in range(SIMS))])
        k = 1 / (N0 * res[:, 0].var())
        k_table[(phi, h)] = k
        print(f"{phi:>5} {str(h):>8} {res[:, 0].std():>8.4f} {k:>6.1f}   "
              f"{np.mean(res[:, 1] >= T_CRIT):>20.3f} {np.mean(res[:, 2] >= T_CRIT):>6.3f}")

  phi  horizon   sd(IC)      k   false positives: clustered  naive


  0.8        5   0.0145   23.7                  0.031  0.138


  0.8       15   0.0193   13.4                  0.025  0.201


  0.8     rest   0.0217   10.6                  0.026  0.238


 0.95        5   0.0166   18.2                  0.032  0.169


 0.95       15   0.0274    6.7                  0.030  0.274


 0.95     rest   0.0372    3.6                  0.027  0.334


 0.99        5   0.0170   17.4                  0.025  0.180


 0.99       15   0.0276    6.6                  0.017  0.245


 0.99     rest   0.0532    1.8                  0.034  0.383


## Step 2: matches needed for 80% power

The t-statistic grows with the square root of the information: mean t is about
`IC * sqrt(n_matches * k)`. Setting that equal to 2.84 (which gives an 80% chance of
clearing t = 2) and solving:

`matches needed = (2.84 / IC)^2 / k`

In [3]:
ICS = (0.02, 0.03, 0.05)
print(f"{'phi':>5} {'horizon':>8}" + "".join(f"{'IC ' + str(ic):>12}" for ic in ICS))
for phi in PHIS:
    for h in HORIZONS:
        k = k_table[(phi, h)]
        print(f"{phi:>5} {str(h):>8}" + "".join(f"{(T_80 / ic) ** 2 / k:>12,.0f}" for ic in ICS))

  phi  horizon     IC 0.02     IC 0.03     IC 0.05
  0.8        5         851         378         136
  0.8       15       1,509         671         241
  0.8     rest       1,907         848         305
 0.95        5       1,107         492         177
 0.95       15       3,030       1,347         485
 0.95     rest       5,574       2,478         892
 0.99        5       1,160         516         186
 0.99       15       3,082       1,370         493
 0.99     rest      11,418       5,075       1,827


## Step 3: check the formula by brute force

Pick one case, compute the predicted matches for 80% power, then simulate that many
matches repeatedly and count how often t clears 2. It should come out close to 80%.

In [4]:
phi, h, beta = 0.95, 5, 0.1
ic_true = np.mean([rank_ic_test(f, target(d, h))[0]
                   for f, d, _ in (simulate(1000, beta, phi, rng) for _ in range(50))])
n80 = int(round((T_80 / ic_true) ** 2 / k_table[(phi, h)]))
hits = [rank_ic_test(f, target(d, h))[1] >= T_CRIT
        for f, d, _ in (simulate(n80, beta, phi, rng) for _ in range(400))]
print(f"beta {beta}, phi {phi}, horizon {h}: true IC {ic_true:.4f}, predicted matches for 80% "
      f"power {n80}, simulated power at that size {np.mean(hits):.1%}")

beta 0.1, phi 0.95, horizon 5: true IC 0.0355, predicted matches for 80% power 352, simulated power at that size 75.0%


## Step 4: what does an IC mean in football terms?

An IC of 0.02 is a figure borrowed from equities, where returns are continuous. Here
the outcome is goals, which are rare and lumpy, so even a sizeable effect gives a
small IC. This table converts the effect size `beta` (how much one standard deviation
of the factor shifts the goal rate) into the IC it produces.

In [5]:
BETAS = (0.05, 0.1, 0.2, 0.3)
phi = 0.95
print(f"phi {phi}.  rows: goal-rate shift per SD of the factor; columns: true rank-IC by horizon")
print(f"{'beta':>6} {'shift':>7}" + "".join(f"{str(h):>9}" for h in HORIZONS))
ic_of = {}
for beta in BETAS:
    row = []
    for h in HORIZONS:
        ic = np.mean([rank_ic_test(f, target(d, h))[0]
                      for f, d, _ in (simulate(1000, beta, phi, rng) for _ in range(40))])
        ic_of[(beta, h)] = ic
        row.append(ic)
    print(f"{beta:>6} {np.exp(beta) - 1:>6.0%}" + "".join(f"{ic:>9.4f}" for ic in row))

phi 0.95.  rows: goal-rate shift per SD of the factor; columns: true rank-IC by horizon
  beta   shift        5       15     rest


  0.05     5%   0.0154   0.0223   0.0182


   0.1    11%   0.0329   0.0439   0.0357


   0.2    22%   0.0646   0.0864   0.0723


   0.3    35%   0.0990   0.1335   0.1143


## Step 5: is rank-IC the best test for goals?

Rank-IC throws information away when the outcome is a rare count: most windows have
zero goals, which become one big tie. The alternative is to test the goal-rate model
directly (a score test for `beta = 0`, the same idea as comparing log-loss of a model
with and without the factor, which the final test's Diebold-Mariano comparison does).
Matches needed for 80% power, same simulated world:

In [6]:
phi, N1, SIMS1 = 0.95, 400, 200
print(f"{'beta':>6}{'rank-IC 5 min':>16}{'rank-IC 15 min':>16}{'rate test':>12}")
for beta in (0.05, 0.1, 0.2):
    out = []
    for h in (5, 15):
        out.append((T_80 / ic_of[(beta, h)]) ** 2 / k_table[(phi, h)])
    z = np.mean([hazard_z(*simulate(N1, beta, phi, rng)) for _ in range(SIMS1)])
    out.append(N1 * (T_80 / z) ** 2)
    print(f"{beta:>6}" + "".join(f"{v:>16,.0f}" for v in out[:2]) + f"{out[2]:>12,.0f}")

  beta   rank-IC 5 min  rank-IC 15 min   rate test


  0.05           1,879           2,426       1,378


   0.1             409             628         321


   0.2             106             163          80


## Step 6: real-world adjustments

The numbers above are best cases. Three things push them up:
- **Multiple testing (BH across the factor family).** With 10 factors and only one
  real, BH at q = 0.05 needs that factor's p below 0.005, so |t| above about 2.81
  instead of 2. Factor between 1 (many real effects) and about 1.65 (one real effect).
- **Locked holdout.** If 30% of matches are locked away, factors are developed and
  first tested on 70%: factor about 1.43.
- **Computer-vision error.** A factor measured with noise has a weaker correlation.
  If the measured factor's reliability is 0.7 (70% of its variance is real signal),
  IC shrinks by `sqrt(0.7)` and matches needed grow by `1 / 0.7`, about 1.43. The
  Stage 1 spot-check (ball recall 54%) suggests ball-based factors could be worse.

Combined worst case: about 1.65 x 1.43 x 1.43, roughly **3.4 times** the base numbers.

In [7]:
t_bh = 2.807                     # two-sided p = 0.005
mult_bh = ((t_bh + 0.8416) / T_80) ** 2
mult = mult_bh * (1 / 0.7) * (1 / 0.7)
print(f"BH multiplier {mult_bh:.2f}, holdout 1.43, CV error 1.43 -> combined {mult:.2f}")
print(f"\nMatches needed (phi 0.95), base / worst case:")
print(f"{'horizon':>8}" + "".join(f"{'IC ' + str(ic):>18}" for ic in ICS))
for h in HORIZONS:
    k = k_table[(0.95, h)]
    print(f"{str(h):>8}" + "".join(f"{(T_80 / ic) ** 2 / k:>8,.0f} / {(T_80 / ic) ** 2 / k * mult:>7,.0f}"
                                  for ic in ICS))

AVAILABLE = 500   # SoccerNet main corpus size, the largest archive considered (access unverified)
print(f"\nWith {AVAILABLE} matches, smallest IC detectable at 80% power (base case, phi 0.95):")
for h in HORIZONS:
    print(f"  {str(h):>5}: IC {T_80 / np.sqrt(AVAILABLE * k_table[(0.95, h)]):.3f}")

BH multiplier 1.65, holdout 1.43, CV error 1.43 -> combined 3.36

Matches needed (phi 0.95), base / worst case:
 horizon           IC 0.02           IC 0.03           IC 0.05
       5   1,107 /   3,725     492 /   1,656     177 /     596
      15   3,030 /  10,195   1,347 /   4,531     485 /   1,631
    rest   5,574 /  18,756   2,478 /   8,336     892 /   3,001

With 500 matches, smallest IC detectable at 80% power (base case, phi 0.95):
      5: IC 0.030
     15: IC 0.049
   rest: IC 0.067


## Step 7: minimum detectable effect for the pre-registered primary test

Decided 2026-10-05: success is stated as a **minimum detectable effect (MDE)** for a
likelihood-based goal-rate test, not as a fixed rank-IC. Two versions compared:
- **Per minute:** factor measured at minute t-1, outcome = goal difference in minute
  t, for every minute.
- **5-minute blocks:** factor measured in the minute before each non-overlapping
  5-minute block, outcome = goal difference within the block.

Both: score test of `beta = 0` in the Poisson goal-rate model (in real data the
baseline expected rate enters as an offset), variance clustered by match, point in
time correct. The more powerful version becomes the primary test.

The MDE is the smallest `beta` with 80% power, reported as the goal-rate shift per
standard deviation of the factor, `exp(beta) - 1`. Base case: one factor, t of 2 or
more. Adjusted case: BH across 5 factors with one real effect (two-sided p of 0.01,
|t| of 2.576 or more), developed on 70% of matches, factor reliability 0.7.

In [8]:
def block_z(f, diff, block):
    starts = np.arange(block, MINUTES - block + 1, block)
    fb = f[:, starts - 1]                                    # factor just before each block
    gd = np.stack([diff[:, s:s + block].sum(axis=1) for s in starts], axis=1)
    u = (fb * gd).sum(axis=1)                                # one score term per match
    return u.sum() / np.sqrt((u ** 2).sum())                 # match-clustered variance


phi, N2 = 0.95, 400
c_hat = {}
for block in (1, 5):
    null_z = np.array([block_z(f, d, block) for f, d, _ in (simulate(N2, 0.0, phi, rng) for _ in range(1000))])
    cs = []
    for beta in (0.05, 0.1, 0.2):
        z = np.mean([block_z(f, d, block) for f, d, _ in (simulate(N2, beta, phi, rng) for _ in range(400))])
        cs.append(z / (beta * np.sqrt(N2)))
    c_hat[block] = np.mean(cs)
    print(f"{block}-minute steps: false positives {np.mean(null_z >= T_CRIT):.3f} (nominal 0.023), "
          f"information per match c = {c_hat[block]:.3f} (by beta: {', '.join(f'{x:.3f}' for x in cs)})")
print(f"per-minute needs {(c_hat[5] / c_hat[1]) ** 2:.0%} of the matches the 5-minute version needs")

t_adj = 2.576 + 0.8416
c1 = c_hat[1]
print(f"\nMDE (80% power) for the per-minute test, goal-rate shift per SD of the factor, phi {phi}:")
print(f"{'matches':>8} {'base':>8} {'adjusted':>10}")
for n in (100, 200, 300, 500, 1000, 2000):
    b_base = T_80 / (c1 * np.sqrt(n))
    b_adj = t_adj / (c1 * np.sqrt(0.7 * n)) / np.sqrt(0.7)
    print(f"{n:>8} {np.exp(b_base) - 1:>8.1%} {np.exp(b_adj) - 1:>10.1%}")

1-minute steps: false positives 0.020 (nominal 0.023), information per match c = 1.529 (by beta: 1.596, 1.535, 1.456)


5-minute steps: false positives 0.019 (nominal 0.023), information per match c = 1.346 (by beta: 1.375, 1.362, 1.303)
per-minute needs 78% of the matches the 5-minute version needs

MDE (80% power) for the per-minute test, goal-rate shift per SD of the factor, phi 0.95:
 matches     base   adjusted
     100    20.4%      37.6%
     200    14.0%      25.3%
     300    11.3%      20.2%
     500     8.7%      15.3%
    1000     6.1%      10.6%
    2000     4.2%       7.4%


## Assumptions (and why they matter)

- **Goal model:** goals arrive independently minute by minute at a rate the factor
  tilts (log-linear). Real goal rates also depend on score and time remaining; the
  final test neutralises against that baseline, so this is the *incremental* effect.
- **Factor dynamics:** AR(1) with `phi` 0.8, 0.95 or 0.99 per minute (half-life about
  3, 14 or 69 minutes). Slower factors carry less independent information per match.
- **Full 90 minutes observed.** Broadcast video loses minutes to replays, close-ups
  and crowd shots, which lowers `k` further.
- **One factor, tested alone.** Section 6 handles the family-size cost roughly.
- **Simulation size:** 1,000 null simulations per cell, so `k` is precise to about
  +-5%. Fixed seed for reproducibility.
- **No market data.** These targets are realised goals (the "decouple" option).
  Market-price targets move continuously and may carry more information per match;
  that needs Track 2 data to quantify and is not covered here.

Conclusions and recommendations are recorded in `decision_log.md` (2026-10-05).